In [ ]:
# Practice Problems Day 9

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 9 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice today's code before the weekly homework. Weekly Homework 5 assumes you have done them. Plan on about 45 minutes.

**Reading:** Geron chapter 9, **From Biological to Artificial Neurons**, and chapter 10, **PyTorch Fundamentals**.

GOALS:

1. Write gradient descent yourself, first for one number and then for a whole neuron.
2. Train a neuron on real data and check it against scikit-learn.
3. Explain in words what a trained neuron learned.

**How much of this should you write yourself?** Problem 1 gives you most of the code, with comments telling you what to write on the missing lines. Problem 2 lists the steps. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for the weekly homework and the exam:

```bash
git add .
git commit -m "Day 9 practice"
git push
```

---

## Problem 1: Gradient descent on one number

**What to copy, what to write:** Most of the code is given. Write the lines the comments ask for.

Before we train a whole neuron, let's do gradient descent on the simplest problem there is. We want the value of $w$ that makes

$$g(w) = (w - 4)^2$$

as small as possible. You already know the answer is $w = 4$, so we can check that gradient descent finds it.

**1a.** By hand, find $g'(w)$ and then $g'(1)$. Then run the cell below and check that autograd agrees with you.

In [ ]:
import torch

w = torch.tensor(1.0, requires_grad=True)

# Write g(w) = (w - 4)^2, then call .backward() on it
g = (w - 4) ** 2
g.backward()

print("g'(1) from autograd:", w.grad)

**1b.** Now gradient descent. Start at $w = 0$ and take 50 steps with a learning rate of 0.1. Where does $w$ end up?

**1c.** Change the learning rate to **1.1** and run it again. What does $w$ do? (Hint: work out by hand where the first step from $w = 0$ lands.)

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.1

for step in range(50):
    # compute g = (w - 4)^2
    g = (w - 4) ** 2

    # call backward to find the derivative
    g.backward()

    # take a step downhill (inside torch.no_grad())
    with torch.no_grad():
        w -= learning_rate * w.grad

    # clear the gradient
    w.grad.zero_()

    if step % 10 == 0:                 # % is the remainder, so this is True for steps 0, 10, 20, ...
        print("step", step, "  w =", round(w.item(), 4))

print("final w:", round(w.item(), 4))

In [ ]:
# 1c: the same loop with a learning rate of 1.1
w = torch.tensor(0.0, requires_grad=True)
learning_rate = 1.1

for step in range(50):
    g = (w - 4) ** 2
    g.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad
    w.grad.zero_()
    if step % 10 == 0:
        print("step", step, "  w =", round(w.item(), 4))

print("final w:", round(w.item(), 4))

**1a.** $g'(w) = 2(w - 4)$, so $g'(1) = -6$. Autograd prints `tensor(-6.)`.

**1b.** $w$ climbs toward 4 and ends at **3.9999** after 50 steps. Each step moves it 20 percent of the remaining distance, since $w - 0.1 \cdot 2(w - 4) = 0.8w + 0.8$.

**1c.** The first step lands at $0 - 1.1 \cdot 2(0 - 4) = 8.8$, which is past 4 by more than it started. The next lands at $-1.76$, and every jump is bigger than the last, so $w$ blows up. The second cell above ends at about $-36{,}398$ after 50 steps. The step is the learning rate times the slope, and when it is too big it overshoots the bottom. This is the red band from the learning rate plot in the notes.

---

## Problem 2: Train a neuron on the penguins (steps outlined)

**What to copy, what to write:** The first cell is written for you, just run it. In the second cell, write the loop yourself. It is the same pattern as the notes, so type it out rather than pasting it.

The data is the penguins from the Day 8 homework. We use the four body measurements to predict a penguin's **sex** (1 for male, 0 for female).

**2a.** Train a neuron with gradient descent: start every weight and the bias at 0, use a learning rate of **0.1**, and take **2000 steps**. Plot the loss at every step, with scikit-learn's loss as a dashed line.

**2b.** Did your neuron reach scikit-learn's loss after 2000 steps? Compare the validation accuracy of your neuron and scikit-learn.

In [ ]:
# RUN THIS. Load, split, scale, tensors, and scikit-learn's model, all like the notes.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

penguins = pd.read_csv("data/penguins.csv").dropna()
measurements = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = penguins[measurements]
y = (penguins["sex"] == "Male").astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
preprocessing = Pipeline([("scale", StandardScaler())])      # the preprocessing pipeline, fit on the training rows only
X_train_scaled = preprocessing.fit_transform(X_train)
X_valid_scaled = preprocessing.transform(X_valid)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
X_valid_t = torch.tensor(X_valid_scaled, dtype=torch.float32)

# scikit-learn's model in a full pipeline, fit on the raw rows
log_reg_pipeline = Pipeline([("prep", preprocessing), ("model", LogisticRegression(C=np.inf))])   # C=np.inf: no regularization
log_reg_pipeline.fit(X_train, y_train)
log_reg = log_reg_pipeline.named_steps["model"]          # the fitted model inside the pipeline
p_sklearn = torch.tensor(log_reg.predict_proba(X_train_scaled)[:, 1], dtype=torch.float32)
loss_sklearn = F.binary_cross_entropy(p_sklearn, y_train_t)

print("training penguins:", X_train_t.shape[0], "   validation penguins:", X_valid_t.shape[0])
print("scikit-learn's loss:", round(loss_sklearn.item(), 4))

In [ ]:
# Step 1 (2a): w (4 zeros) and b (1 zero), both with requires_grad=True. learning_rate = 0.1, and an empty losses list
w = torch.zeros(4, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
learning_rate = 0.1
losses = []

# Step 2 (2a): the loop, 2000 steps: predict, loss (save it), backward, update inside torch.no_grad(), clear
for step in range(2000):
    p = torch.sigmoid(X_train_t @ w + b)
    loss = F.binary_cross_entropy(p, y_train_t)
    losses.append(loss.item())
    loss.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    w.grad.zero_()
    b.grad.zero_()
print("neuron loss after 2000 steps:", round(losses[-1], 4))

# Step 3 (2a): plot losses, with scikit-learn's loss as a dashed line: plt.axhline(loss_sklearn.item(), linestyle="--")
plt.figure(figsize=(6, 4))
plt.plot(losses, label="our neuron")
plt.axhline(loss_sklearn.item(), color="gray", linestyle="--", label="scikit-learn")
plt.xlabel("step")
plt.ylabel("log loss on the training penguins")
plt.legend()
plt.grid()
plt.show()

# Step 4 (2b): predict the validation penguins with your neuron (threshold 0.5), then print both validation accuracies
#              scikit-learn's predictions are log_reg_pipeline.predict(X_valid)
with torch.no_grad():
    p_valid = torch.sigmoid(X_valid_t @ w + b)
y_pred_neuron = (p_valid.numpy() >= 0.5).astype(int)
y_pred_sklearn = log_reg_pipeline.predict(X_valid)

print("neuron validation accuracy:      ", round(accuracy_score(y_valid, y_pred_neuron), 3))
print("scikit-learn validation accuracy:", round(accuracy_score(y_valid, y_pred_sklearn), 3))
print("share where they agree:", round((y_pred_neuron == y_pred_sklearn).mean(), 3))


**2a.** The neuron's loss after 2000 steps is **0.2247** and scikit-learn's is **0.2202**. On the plot the neuron's curve is still sloping down a little at step 2000.

**2b.** No, not quite. It is close but still going. Validation accuracy is 0.893 for the neuron and 0.881 for scikit-learn. They disagree on exactly one of the 84 validation penguins, and that one penguin is the whole difference (1/84 is about 0.012). That does not make the unfinished neuron better, one penguin is noise. A bigger learning rate or more steps would finish the job (see the optional exploration).

---

## Problem 3: What did the neuron learn? (words)

Run the cell below. It prints your neuron's four weights next to the name of each measurement.

**3a.** Which measurement pushes the neuron most strongly toward "male"?

**3b.** Explain to a friend who took DATA 101 but not this class what your neuron does with a new penguin, from its four measurements to the word "male" or "female". Use the words weight, sum, sigmoid and threshold, and no code. Three or four sentences.

In [ ]:
# the weights from Problem 2, one per measurement
print(pd.Series(w.detach().numpy(), index=measurements).round(3))
print("bias:", round(b.item(), 3))

**3a.** Body mass (3.901) and bill depth (3.802) have by far the biggest weights, so heavier penguins with deeper bills get pushed hardest toward "male". Bill length is small and positive (0.505) and flipper length is small and negative (-0.297).

**3b.** Your answer will be different, here is one way to say it: "The model gives each of the four measurements a weight, which says how much that measurement counts and in which direction. For a new penguin it multiplies each scaled measurement by its weight and adds them up, plus one extra number. The sigmoid turns that sum into a probability between 0 and 1 that the penguin is male. If the probability is at least 0.5 (the threshold) it says male, otherwise female."

---

## Optional exploration

Only if you are curious. Nothing here is needed for the weekly homework.

**A.** Train the penguin neuron with the learning rates 0.01, 0.1, 1.0 and 20, and plot all four loss curves on one plot. Which one would you use? What goes wrong with 20?

**B.** Start the weights at random numbers, `torch.randn(4, requires_grad=True)`, instead of zeros. Does the neuron end up at the same loss?

In [ ]:
def train_neuron(learning_rate):
    w = torch.zeros(4, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    losses = []
    for step in range(2000):
        p = torch.sigmoid(X_train_t @ w + b)
        loss = F.binary_cross_entropy(p, y_train_t)
        losses.append(loss.item())
        loss.backward()
        with torch.no_grad():
            w -= learning_rate * w.grad
            b -= learning_rate * b.grad
        w.grad.zero_()
        b.grad.zero_()
    return losses

plt.figure(figsize=(7, 4))
for lr in [0.01, 0.1, 1.0, 20.0]:
    lr_losses = train_neuron(lr)
    print("learning rate", lr, "   loss after 2000 steps:", round(lr_losses[-1], 4))
    plt.plot(lr_losses, label="learning rate " + str(lr))
plt.axhline(loss_sklearn.item(), color="gray", linestyle="--", label="scikit-learn")
plt.xlabel("step")
plt.ylabel("log loss")
plt.ylim(0.15, 0.75)
plt.legend()
plt.grid()
plt.show()

**A.** Loss after 2000 steps: 0.01 gives 0.3072, 0.1 gives 0.2247, 1.0 gives **0.2202** (scikit-learn's loss), and 20 ends at 0.4023, worse than 0.1, because each step overshoots the bottom and the loss jumps around instead of going down. 1.0 is the best choice here.

**B.** Yes. From random weights it still ends near scikit-learn's loss (with a learning rate that works), because the log loss for logistic regression has only one bottom.